# LNN vs DeLaN vs VanillaNN — Spring Pendulum (2-DOF)

This notebook compares three models on the spring pendulum — a 2-DOF coupled
nonlinear system — to show where the additional structure of DeLaN pays off
over the vanilla LNN.

| Model | Lagrangian structure | Mass matrix $M(q)$ | Potential $V(q)$ |
|-------|---------------------|---------------------|------------------|
| **VanillaNN** | none | — | — |
| **LNN** | unstructured $L_\theta(q,\dot{q})$ | implicit Hessian, not guaranteed SPD | not separated |
| **DeLaN** | $L = T - V$, $T = \frac{1}{2}\dot{q}^\top M(q)\dot{q}$ | **SPD by Cholesky** | **$\geq 0$ by softplus** |

Both LNN and DeLaN conserve energy by construction and outperform VanillaNN in
extrapolation. On a multi-DOF system, DeLaN's guaranteed positive-definiteness
prevents the instabilities that arise when the LNN's implicit mass matrix becomes
near-singular during rollout.


In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
from systems import SpringPendulum
from models import LNN, DeLaN, VanillaNN
from utils.training import train_dynamics_model

SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
COLORS = {"True": "#2c3e50", "DeLaN": "#2ecc71", "LNN": "#3498db", "VanillaNN": "#e74c3c"}

## Data

Same system and initial conditions as `delan_vs_vanilla.ipynb`:
spring pendulum $(r, \theta)$ with $g=10$, $k=10$, $r_0=1$, starting from
$q_0=(1.1,\,0.5)$, $\dot{q}_0=(0,0)$.

**80 training samples** from $t\in[0,3\,\text{s}]$, RK4 rollout evaluated
over $[0, 8\,\text{s}]$ in Cartesian $(x, y)$.


In [ ]:
sp    = SpringPendulum(g=10.0, k=10.0, r0=1.0)
Q0, QDOT0 = np.array([1.1, 0.5]), np.array([0.0, 0.0])
T_TRAIN, T_EVAL = (0.0, 3.0), (0.0, 8.0)
N_TRAIN, N_EVAL = 80, 800

train_ds  = sp.generate_dataset(Q0, QDOT0, T_TRAIN, N_TRAIN)
test_traj = sp.rollout(Q0, QDOT0, T_EVAL, N_EVAL)
t_eval    = test_traj["t"]
xy_true   = test_traj["xy"]
q_true    = test_traj["q"]
qd_true   = test_traj["qdot"]
E_true    = sp.total_energy(q_true, qd_true)

## Training All Three Models

All models share the same architecture (3 hidden layers, 64 units) and are
trained for 2500 epochs on the same MSE acceleration loss.

The difference is entirely in how $\hat{\ddot{q}}$ is computed:
- **VanillaNN**: direct MLP output
- **LNN**: Euler-Lagrange applied to an unstructured scalar $L_\theta(q, \dot{q})$
- **DeLaN**: Euler-Lagrange applied to the structured $T - V$ decomposition with SPD $M(q)$


In [ ]:
EPOCHS = 2_500

models = {
    "DeLaN":     DeLaN(    n_dof=2, hidden_dim=64, n_layers=3, activation=nn.Softplus),
    "LNN":       LNN(      n_dof=2, hidden_dim=64, n_layers=3),
    "VanillaNN": VanillaNN(n_dof=2, hidden_dim=64, n_layers=3),
}
for name, model in models.items():
    print(f"\nTraining {name} ...")
    hist = train_dynamics_model(model, train_ds, n_epochs=EPOCHS,
                                batch_size=32, device=DEVICE, verbose=True, log_every=500)
    print(f"  final loss={hist['train_loss'][-1]:.4e}  t={hist['wall_time']:.1f}s")

In [ ]:
def rk4(model, q0, qdot0, t_array, device="cpu"):
    q, qdot = q0.copy(), qdot0.copy()
    qs = [q.copy()]
    def acc(qi, qdi):
        qt  = torch.tensor(qi[None],  dtype=torch.float32, device=device)
        qdt = torch.tensor(qdi[None], dtype=torch.float32, device=device)
        with torch.enable_grad():
            return model.predict_acceleration(qt, qdt).detach().cpu().numpy()[0]
    for dt in np.diff(t_array):
        k1v = qdot;              k1a = acc(q,              qdot)
        k2v = qdot + .5*dt*k1a; k2a = acc(q + .5*dt*k1v, qdot + .5*dt*k1a)
        k3v = qdot + .5*dt*k2a; k3a = acc(q + .5*dt*k2v, qdot + .5*dt*k2a)
        k4v = qdot +    dt*k3a; k4a = acc(q +    dt*k3v, qdot +    dt*k3a)
        q    = q    + (dt/6)*(k1v + 2*k2v + 2*k3v + k4v)
        qdot = qdot + (dt/6)*(k1a + 2*k2a + 2*k3a + k4a)
        qs.append(q.copy())
    return np.array(qs)

_model_names = ["DeLaN", "LNN", "VanillaNN"]
split = T_TRAIN[1]
in_m, out_m = t_eval <= split, t_eval > split

preds = {}
for name, model in models.items():
    model.eval()
    q_pred  = np.clip(rk4(model, Q0, QDOT0, t_eval, DEVICE), -20, 20)
    qd_pred = np.gradient(q_pred, t_eval, axis=0)
    xy_pred = SpringPendulum.polar_to_xy(q_pred)
    E_pred  = sp.total_energy(q_pred, qd_pred)
    rmse_in  = float(np.sqrt(np.mean((xy_pred[in_m]  - xy_true[in_m])**2)))
    rmse_out = float(np.sqrt(np.mean((xy_pred[out_m] - xy_true[out_m])**2)))
    e_drift  = float(np.mean(np.abs(E_pred - E_true[0]) / (np.abs(E_true[0]) + 1e-8)))
    preds[name] = {"xy": xy_pred, "E": E_pred, "interp": rmse_in, "extrap": rmse_out, "edrift": e_drift}

print(f"{'Model':<12} {'Interp RMSE':>13} {'Extrap RMSE':>13} {'E-drift':>10}")
print("-" * 52)
for name, p in preds.items():
    print(f"{name:<12} {p['interp']:>13.4f} {p['extrap']:>13.4f} {p['edrift']:>10.4f}")
for name in ["LNN", "VanillaNN"]:
    gain = preds[name]["extrap"] / (preds["DeLaN"]["extrap"] + 1e-12)
    print(f"DeLaN extrap improvement over {name}: {gain:.1f}×")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
fig.suptitle(f"LNN vs DeLaN vs VanillaNN — Spring Pendulum 2-DOF\n"
             f"{N_TRAIN} training pts from [0, 3 s]  |  shaded = extrapolation",
             fontsize=12, fontweight="bold")

for ax, ci, lbl in zip(axes[:2], [0, 1], ["x (m)", "y (m)"]):
    ax.plot(t_eval, xy_true[:, ci], color=COLORS["True"], lw=1, ls="--",
            label="Ground truth", zorder=5)
    for name in _model_names:
        ax.plot(t_eval, np.clip(preds[name]["xy"][:, ci], -5, 5),
                color=COLORS[name], lw=1.8, label=f"{name}  (extrap={preds[name]['extrap']:.4f})")
    ax.axvline(split, color="gray", ls=":", lw=1.4, label="train end")
    ax.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
    ax.set_ylabel(lbl, fontsize=11); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[2]
for name in _model_names:
    err = np.sqrt(np.sum((np.clip(preds[name]["xy"], -5, 5) - xy_true)**2, axis=1))
    ax.semilogy(t_eval, err + 1e-6, color=COLORS[name], lw=1.6, label=name)
ax.axvline(split, color="gray", ls=":", lw=1.4)
ax.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
ax.set_ylabel("‖xy error‖  (log)", fontsize=11); ax.set_xlabel("Time (s)", fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "lnn_vs_delan.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Energy conservation — the hierarchy DeLaN < LNN < VanillaNN in drift
fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(E_true[0], color=COLORS["True"], lw=1.2, ls="--", label=f"True E = {E_true[0]:.3f}")
for name in _model_names:
    ax.plot(t_eval, preds[name]["E"], color=COLORS[name], lw=1.6,
            label=f"{name}  (drift={preds[name]['edrift']:.4f})")
ax.axvline(split, color="gray", ls=":", lw=1.4, label="train end")
ax.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
ax.set_xlabel("t (s)"); ax.set_ylabel("Total energy E = T + V", fontsize=11)
ax.set_title("Energy conservation — DeLaN < LNN < VanillaNN", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "lnn_vs_delan_energy.png"), dpi=150, bbox_inches="tight")
plt.show()